<a href="https://colab.research.google.com/github/Frezga/case-method-SETM-asosiasi/blob/main/Tugas_DataMining_SETM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q prettytable

In [2]:
import time
import pandas as pd
from itertools import combinations
from prettytable import PrettyTable

# 1. KONFIGURASI PARAMETER
FILE_PRODUCTS     = "Products.csv"
FILE_TRANSACTIONS = "Transactions.csv"
TARGET_OUTLET     = "SHOP001"
START_DATE        = "2025-01-01"
END_DATE          = "2025-01-31"

MIN_SUPPORT       = 0.08   # 8% dari total transaksi
MIN_CONFIDENCE    = 0.50   # 50%

# 2. PREPROCESSING & PEMBENTUKAN RELASI R1
def make_full_name(row):
    variant = str(row["Variant"]).strip() if pd.notna(row["Variant"]) else "-"
    name = str(row["ProductName"]).strip()
    return name if variant in ["-", ""] else f"{name} ({variant})"

def get_time_slot(time_str):
    hour = int(str(time_str).split(":")[0])
    if hour < 11:
        return "Sesi_Pagi"
    elif hour < 15:
        return "Sesi_Siang"
    elif hour < 18:
        return "Sesi_Sore"
    return "Sesi_Malam"

def prepare_r1():
    df_prod = pd.read_csv(FILE_PRODUCTS)
    df_tx = pd.read_csv(FILE_TRANSACTIONS)

    # Pemetaan nama produk lengkap (termasuk varian) -> Kategori Menu
    df_prod["FullName"] = df_prod.apply(make_full_name, axis=1)
    prod_map_cat = dict(zip(df_prod["FullName"], df_prod["Category"]))

    # Filter subset: Outlet SHOP001, Januari 2025, transaksi multi-item (2 item)
    df_sub = df_tx[
        (df_tx["Outlet"] == TARGET_OUTLET) &
        (df_tx["Date"] >= START_DATE) &
        (df_tx["Date"] <= END_DATE) &
        (df_tx["TotalItem"] == 2)
    ].copy()

    # Bentuk tabel relasional datar <TID, Item> untuk input algoritma SETM
    rows = []
    for _, row in df_sub.iterrows():
        tid = str(row["ReceiptNumber"]).strip()
        raw_items = [i.strip() for i in str(row["Items"]).split(",") if i.strip()]

        features = set()
        for it in raw_items:
            if it in prod_map_cat:
                features.add("Menu_" + prod_map_cat[it])
        features.add(get_time_slot(row["Time"]))
        features.add("Bayar_" + str(row["PaymentMethod"]).strip())

        for feat in sorted(features):
            rows.append((tid, feat))

    # Urutkan berdasarkan (TID, Item) dan simpan sebagai bukti dataset bersih
    df_r1 = pd.DataFrame(rows, columns=["TID", "Item"]).sort_values(by=["TID", "Item"]).reset_index(drop=True)
    df_r1.to_csv("dataset_umkm_setm.csv", index=False)

    R1 = list(df_r1.itertuples(index=False, name=None))
    total_tx = df_sub["ReceiptNumber"].nunique()
    return R1, total_tx

# 3. FUNGSI INTI ALGORITMA SETM
def sort_count_and_prune(C_bar, min_sup_count):
    """
    Fase Dual-Sorting SETM:
    1. Sort C_bar berdasarkan Itemset agar itemset identik bersebelahan.
    2. Hitung frekuensi kemunculan tiap itemset (linear scan).
    3. Eliminasi itemset < min_sup_count -> hasilkan L_k.
    4. Filter C_bar menjadi L_bar, lalu Sort kembali berdasarkan (TID, Itemset).
    """
    # SORT 1: Urutkan berdasarkan Itemset
    C_bar_sorted = sorted(C_bar, key=lambda x: x[1])

    counts = {}
    for _, itemset in C_bar_sorted:
        counts[itemset] = counts.get(itemset, 0) + 1

    L_k = {itemset: cnt for itemset, cnt in counts.items() if cnt >= min_sup_count}
    L_bar = [row for row in C_bar_sorted if row[1] in L_k]

    # SORT 2: Urutkan kembali berdasarkan TID untuk Merge-Join iterasi berikutnya
    L_bar.sort(key=lambda x: (x[0], x[1]))
    return L_k, L_bar

def merge_join_setm(L_bar, R1):
    """
    Fase Merge-Join SETM:
    Menggabungkan L_bar (terurut TID) dengan R1 (terurut TID)
    dengan syarat p.TID == q.TID dan p.itemset[-1] < q.item.
    """
    C_bar = []
    i, j = 0, 0
    n, m = len(L_bar), len(R1)

    while i < n and j < m:
        tid_l, tid_r = L_bar[i][0], R1[j][0]

        if tid_l < tid_r:
            i += 1
        elif tid_l > tid_r:
            j += 1
        else:
            i_end = i
            while i_end < n and L_bar[i_end][0] == tid_l:
                i_end += 1
            j_end = j
            while j_end < m and R1[j_end][0] == tid_r:
                j_end += 1

            for p in range(i, i_end):
                itemset = L_bar[p][1]
                last_item = itemset[-1]
                for q in range(j, j_end):
                    item_r = R1[q][1]
                    if last_item < item_r:
                        C_bar.append((tid_l, itemset + (item_r,)))

            i, j = i_end, j_end

    return C_bar

def run_setm(R1, total_tx, min_support):
    min_sup_count = max(2, int(round(min_support * total_tx)))
    all_frequent = {}

    print(f"Outlet & Periode     : {TARGET_OUTLET} ({START_DATE} s.d. {END_DATE})")
    print(f"Total Transaksi (N)  : {total_tx}")
    print(f"Total Baris Relasi R1: {len(R1)}")
    print(f"Minimum Support      : {min_support*100:.0f}% (>= {min_sup_count} transaksi)\n")

    # Pass k = 1
    k = 1
    C_bar = [(tid, (item,)) for tid, item in R1]
    L_k, L_bar = sort_count_and_prune(C_bar, min_sup_count)

    while L_k:
        all_frequent[k] = L_k

        tbl = PrettyTable()
        tbl.title = f"Frequent {k}-Itemset (L{k}) | Kandidat C_bar{k}: {len(C_bar)} baris"
        tbl.field_names = ["Itemset", "Count", "Support (%)"]
        for itemset, cnt in sorted(L_k.items(), key=lambda x: -x[1]):
            tbl.add_row([", ".join(itemset), cnt, f"{(cnt/total_tx)*100:.2f}%"])
        print(tbl)
        print()

        # Pass k + 1
        k += 1
        C_bar = merge_join_setm(L_bar, R1)
        if not C_bar:
            break
        L_k, L_bar = sort_count_and_prune(C_bar, min_sup_count)

    return all_frequent

# ==========================================
# 4. PEMBANGKITAN ATURAN ASOSIASI
# ==========================================
def generate_rules(all_frequent, total_tx, min_confidence):
    support_lookup = {}
    for Lk in all_frequent.values():
        support_lookup.update(Lk)

    rules = []
    for k, Lk in all_frequent.items():
        if k < 2:
            continue
        for itemset, cnt_ab in Lk.items():
            for r in range(1, len(itemset)):
                for ant in combinations(itemset, r):
                    cons = tuple(x for x in itemset if x not in ant)
                    conf = cnt_ab / support_lookup[ant]
                    lift = conf / (support_lookup[cons] / total_tx)

                    if conf >= min_confidence and lift > 1.0:
                        rules.append((ant, cons, cnt_ab, cnt_ab / total_tx, conf, lift))

    rules.sort(key=lambda x: (-x[5], -x[4]))
    return rules

# 5. EKSEKUSI UTAMA
if __name__ == "__main__":
    start_time = time.perf_counter()

    R1, total_tx = prepare_r1()
    all_frequent = run_setm(R1, total_tx, MIN_SUPPORT)
    rules = generate_rules(all_frequent, total_tx, MIN_CONFIDENCE)

    elapsed_ms = (time.perf_counter() - start_time) * 1000

    tbl_rules = PrettyTable()
    tbl_rules.title = f"Association Rules SETM (Min Conf: {MIN_CONFIDENCE*100:.0f}%, Lift > 1.0)"
    tbl_rules.field_names = ["No", "Antecedent (Jika)", "Consequent (Maka)", "Count", "Support", "Confidence", "Lift"]

    for idx, (ant, cons, cnt, sup, conf, lift) in enumerate(rules, 1):
        tbl_rules.add_row([
            idx,
            ", ".join(ant),
            ", ".join(cons),
            cnt,
            f"{sup*100:.2f}%",
            f"{conf*100:.2f}%",
            f"{lift:.2f}"
        ])

    print(tbl_rules)
    print(f"\nTotal aturan ditemukan   : {len(rules)}")
    print(f"Total waktu eksekusi SETM: {elapsed_ms:.2f} ms")

Outlet & Periode     : SHOP001 (2025-01-01 s.d. 2025-01-31)
Total Transaksi (N)  : 406
Total Baris Relasi R1: 1526
Minimum Support      : 8% (>= 32 transaksi)

+-------------------------------------------------------+
| Frequent 1-Itemset (L1) | Kandidat C_bar1: 1526 baris |
+------------------------+---------+--------------------+
|        Itemset         |  Count  |    Support (%)     |
+------------------------+---------+--------------------+
|       Bayar_BCA        |   323   |       79.56%       |
|      Menu_Coffee       |   242   |       59.61%       |
|    Menu_Menu Gedhe     |   153   |       37.68%       |
|       Sesi_Malam       |   130   |       32.02%       |
|       Menu_Other       |   112   |       27.59%       |
|        Menu_Tea        |   102   |       25.12%       |
|       Sesi_Pagi        |    98   |       24.14%       |
|       Sesi_Siang       |    92   |       22.66%       |
|       Sesi_Sore        |    86   |       21.18%       |
|       Bayar_Cash       |  